# CS9 Spatial Confluence: HOLC × Firearm Homicide × Lead Exposure × Incarceration

**Case study:** *Spatial Confluence — HOLC Redlining, Firearm Homicide, Lead Exposure, and Incarceration Across Six Cities*  
**Equations:** eq:9.1–eq:9.10 (compounding capacity-reduction chain)  
**Tier:** 1  

## Environment (preferred)

```bash
conda env create -f Paper/scripts/spatial_env.yml
conda activate spatial_cs9
jupyter notebook Paper/scripts/eq47_51_spatial_overlay.ipynb
```

Data must be fetched first:
```bash
python Paper/scripts/fetch_spatial_data.py --data-dir Paper/data/spatial
```

If the conda env is not active the **setup cell below** will attempt `pip install`.
The notebook degrades gracefully to synthetic data when spatial packages are absent.

## Cities
Memphis TN · Detroit MI · Nashville TN · Baltimore MD · Washington DC · Milwaukee WI

## Output files
- `Paper/data/spatial/merged_tract_panel_<city>.parquet` (one per city)
- `Paper/data/spatial/pooled_panel.parquet`
- `Paper/figures/spatial/cs9_overlay_<city>.png` (four-panel choropleth, one per city)
- `Paper/figures/spatial/cs9_lisa_<city>.png` (bivariate LISA cluster map, one per city)
- `Paper/figures/spatial/cs9_pooled_stats.png` (forest plot of pooled effect sizes)

In [10]:
# ── Environment setup ─────────────────────────────────────────────────────
# Run this cell first.  It installs missing spatial packages via pip when the
# spatial_cs9 conda environment is not active.  Conda is preferred because
# geopandas/osmnx depend on binary C/GDAL libraries; pip may fail on Windows
# or minimal CI images.  When pip also fails the notebook runs in
# 'synthetic-data / statistics-skipped' mode so nbconvert does not abort.

import importlib, subprocess, sys

SPATIAL_PKGS = [
    "geopandas",
    "contextily",
    "folium",
    "libpysal",
    "esda",
    "spreg",
    "pyarrow",
    "shapely",
    "requests",
    "tqdm",
]

missing = [pkg for pkg in SPATIAL_PKGS if importlib.util.find_spec(pkg) is None]

if missing:
    print(f"Installing missing packages via pip: {missing}")
    try:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "--quiet"] + missing,
            stderr=subprocess.STDOUT,
        )
        print("pip install complete. You may need to restart the kernel if imports still fail.")
    except subprocess.CalledProcessError:
        print(
            f"pip install failed for: {missing}\n"
            "Notebook will run in degraded mode (synthetic data, no spatial statistics).\n"
            "Recommended fix:\n"
            "  conda env create -f Paper/scripts/spatial_env.yml\n"
            "  conda activate spatial_cs9\n"
        )
else:
    print("All spatial packages present — full pipeline mode.")

All spatial packages present — full pipeline mode.


In [11]:
from __future__ import annotations

import os
import types
import warnings
from pathlib import Path

# ── Standard library (always available) ───────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.gridspec import GridSpec
from scipy import stats as scipy_stats

# ── geopandas / shapely (graceful degradation) ────────────────────────────
try:
    import geopandas as gpd
    from shapely.geometry import Point
    GEOPANDAS_AVAILABLE = True
    print("geopandas OK")
except ImportError:
    warnings.warn(
        "geopandas / shapely not found.  Run the setup cell or:\n"
        "  conda env create -f Paper/scripts/spatial_env.yml && conda activate spatial_cs9\n"
        "Notebook will use synthetic data and skip all spatial joins."
    )
    GEOPANDAS_AVAILABLE = False
    # Minimal stubs so the rest of the notebook can be parsed
    gpd = types.SimpleNamespace(
        GeoDataFrame=pd.DataFrame,
        read_file=lambda *a, **kw: pd.DataFrame(),
        points_from_xy=lambda *a, **kw: [],
        sjoin=lambda *a, **kw: pd.DataFrame(),
    )
    class Point:  # type: ignore[no-redef]
        def __init__(self, *args): pass

# ── contextily (graceful degradation) ─────────────────────────────────────
try:
    import contextily as ctx
    CONTEXTILY_AVAILABLE = True
    print("contextily OK")
except ImportError:
    warnings.warn("contextily not found; basemap tiles will be skipped.")
    CONTEXTILY_AVAILABLE = False
    ctx = types.SimpleNamespace(
        add_basemap=lambda *a, **kw: None,
        providers=types.SimpleNamespace(
            CartoDB=types.SimpleNamespace(Positron=None)
        ),
    )

# ── libpysal / esda / spreg (graceful degradation) ────────────────────────
try:
    import libpysal.weights as lps_weights
    from esda.moran import Moran, Moran_BV, Moran_Local_BV
    import spreg
    SPATIAL_STATS_AVAILABLE = True
    print("libpysal / esda / spreg OK")
except ImportError:
    warnings.warn("libpysal / esda / spreg not found; spatial statistics will be skipped.")
    SPATIAL_STATS_AVAILABLE = False
    lps_weights = None  # type: ignore[assignment]

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

geopandas OK
contextily OK
libpysal / esda / spreg OK


In [12]:
# ── Paths (cwd-independent) ─────────────────────────────────────────────
# `Path("../..")` breaks: Jupyter's cwd is often the *repo root*, not
# `Paper/scripts/`, so `../../` escapes the project.  Walk upward from cwd
# until we find the manuscript (or Paper/data), then anchor paths there.
def _resolve_repo_root() -> Path:
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / "Paper" / "Redefining_Racism.tex").is_file():
            return d
        if (d / "Paper" / "data").is_dir():
            return d
    return here  # last resort: cwd

REPO_ROOT = _resolve_repo_root()
DATA_DIR  = REPO_ROOT / "Paper" / "data" / "spatial"
FIG_DIR   = REPO_ROOT / "Paper" / "figures" / "spatial"
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
print(f"REPO_ROOT = {REPO_ROOT}")
print(f"DATA_DIR  = {DATA_DIR}")
print(f"FIG_DIR   = {FIG_DIR}")

TARGET_CRS = "EPSG:4326"
PLOT_CRS   = "EPSG:3857"   # Web Mercator for contextily basemap

# ── City catalogue ─────────────────────────────────────────────────────────
CITIES = [
    {"id": "memphis_tn",    "label": "Memphis TN",    "ppi_level": "county",
     "fips_county": "47157", "state": "TN"},
    {"id": "detroit_mi",    "label": "Detroit MI",    "ppi_level": "tract",
     "fips_county": "26163", "state": "MI"},
    {"id": "nashville_tn",  "label": "Nashville TN",  "ppi_level": "county",
     "fips_county": "47037", "state": "TN"},
    {"id": "baltimore_md",  "label": "Baltimore MD",  "ppi_level": "tract",
     "fips_county": "24510", "state": "MD"},
    {"id": "washington_dc", "label": "Washington DC", "ppi_level": "tract",
     "fips_county": "11001", "state": "DC"},
    {"id": "milwaukee_wi",  "label": "Milwaukee WI",  "ppi_level": "county",
     "fips_county": "55079", "state": "WI"},
]

HOLC_COLORS = {"A": "#76a865", "B": "#7cb9e8", "C": "#ffff99", "D": "#d9534f"}

print("Environment ready. Cities:", [c["label"] for c in CITIES])

REPO_ROOT = /Users/emmanuel/Documents/Theory/Redefining_racism
DATA_DIR  = /Users/emmanuel/Documents/Theory/Redefining_racism/Paper/data/spatial
FIG_DIR   = /Users/emmanuel/Documents/Theory/Redefining_racism/Paper/figures/spatial
Environment ready. Cities: ['Memphis TN', 'Detroit MI', 'Nashville TN', 'Baltimore MD', 'Washington DC', 'Milwaukee WI']


---
## Phase B — Per-City Spatial Join Pipeline

In [13]:
def load_holc(city: dict):
    """Load HOLC GeoJSON for a city. Returns None when geopandas is unavailable."""
    if not GEOPANDAS_AVAILABLE:
        return None
    path = DATA_DIR / f"holc_{city['id']}.geojson"
    if not path.exists():
        print(f"  [HOLC] {city['label']}: file not found — run fetch_spatial_data.py")
        return None
    gdf = gpd.read_file(path)
    if gdf.empty:
        print(f"  [HOLC] {city['label']}: empty GeoDataFrame (stub)")
        return None
    gdf = gdf.to_crs(TARGET_CRS)
    # Normalise grade column
    for col in ["holc_grade", "grade", "HOLC_grade", "Grade"]:
        if col in gdf.columns:
            gdf = gdf.rename(columns={col: "holc_grade"})
            break
    if "holc_grade" not in gdf.columns:
        gdf["holc_grade"] = "D"   # fallback
    gdf["holc_d_flag"] = (gdf["holc_grade"] == "D").astype(int)
    print(f"  [HOLC] {city['label']}: {len(gdf)} polygons")
    return gdf


def load_acs(city: dict):
    """Load ACS CSV for a city."""
    path = DATA_DIR / f"acs_{city['id']}.csv"
    if not path.exists():
        print(f"  [ACS]  {city['label']}: file not found")
        return None
    df = pd.read_csv(path, comment="#")
    if df.empty:
        return None
    rename_map = {
        "B02001_001E": "acs_total_pop",
        "B02001_002E": "acs_white_pop",
        "B02001_003E": "acs_black_pop",
        "B19013_001E": "acs_median_income",
    }
    df = df.rename(columns=rename_map)
    for col in ["acs_total_pop", "acs_white_pop", "acs_black_pop", "acs_median_income"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    df["acs_pct_black"] = (
        df["acs_black_pop"] / df["acs_total_pop"].replace(0, np.nan) * 100
    )
    df["acs_pct_white"] = (
        df["acs_white_pop"] / df["acs_total_pop"].replace(0, np.nan) * 100
    )
    if "GEO_ID" in df.columns:
        df["GEOID"] = df["GEO_ID"].str.replace("1400000US", "", regex=False).str.strip()
    elif {"state", "county", "tract"}.issubset(df.columns):
        df["GEOID"] = (
            df["state"].astype(str).str.zfill(2)
            + df["county"].astype(str).str.zfill(3)
            + df["tract"].astype(str).str.zfill(6)
        )
    print(f"  [ACS]  {city['label']}: {len(df)} tracts")
    return df


def load_ejscreen(city: dict):
    """Load EJScreen CSV for a city."""
    path = DATA_DIR / f"ejscreen_{city['id']}.csv"
    if not path.exists():
        print(f"  [EJS]  {city['label']}: file not found")
        return None
    df = pd.read_csv(path, comment="#", encoding="latin-1", low_memory=False)
    if df.empty:
        return None
    rename_map = {
        "ID":     "GEOID",
        "LDPNT":  "lead_paint_index",
        "PTRAF":  "highway_buffer_proxy",
        "CANCR":  "cancer_risk_index",
        "RESP":   "respiratory_index",
    }
    df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})
    if "GEOID" in df.columns:
        df["GEOID"] = df["GEOID"].astype(str).str.strip().str.zfill(11)
    for col in ["lead_paint_index", "highway_buffer_proxy"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    print(f"  [EJS]  {city['label']}: {len(df)} tracts")
    return df


def load_gva(city: dict):
    """Load GVA incidents CSV for a city (stub-aware)."""
    path = DATA_DIR / f"gva_incidents_{city['id']}.csv"
    if not path.exists():
        print(f"  [GVA]  {city['label']}: file not found")
        return None
    text = path.read_text()
    if "DATA_GAP" in text:
        print(f"  [GVA]  {city['label']}: DATA_GAP stub — returning None")
        return None
    df = pd.read_csv(path, comment="#")
    print(f"  [GVA]  {city['label']}: {len(df)} incidents")
    return df


def load_cdc_wonder(city: dict):
    """Load CDC WONDER fallback firearm mortality (county-level stub-aware)."""
    path = DATA_DIR / f"cdc_wonder_firearm_{city['id']}.csv"
    if not path.exists():
        return None
    text = path.read_text()
    if "DATA_GAP" in text:
        return None
    return pd.read_csv(path, comment="#")


def load_ppi(city: dict):
    """Load PPI incarceration origin CSV (tract/county level, stub-aware)."""
    path = DATA_DIR / f"ppi_{city['id']}.csv"
    if not path.exists():
        print(f"  [PPI]  {city['label']}: file not found")
        return None
    text = path.read_text()
    if "DATA_GAP" in text:
        print(f"  [PPI]  {city['label']}: DATA_GAP stub — returning None")
        return None
    df = pd.read_csv(path, comment="#")
    df["ppi_data_level"] = city["ppi_level"]
    print(f"  [PPI]  {city['label']}: {len(df)} rows ({city['ppi_level']}-level)")
    return df

In [14]:
def build_tract_grid(city: dict):
    """
    Load Census tract boundaries via the Census TIGER WMS API.
    Returns an empty DataFrame when geopandas is unavailable (triggers synthetic fallback).
    """
    if not GEOPANDAS_AVAILABLE:
        return pd.DataFrame({"GEOID": []})

    import requests
    county_fips = city["fips_county"]
    state_fips  = county_fips[:2]
    county_code = county_fips[2:]
    url = (
        "https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/"
        "tigerWMS_Census2020/MapServer/8/query"
    )
    params = {
        "where": f"STATE='{state_fips}' AND COUNTY='{county_code}'",
        "outFields": "GEOID,AREALAND,AREAWATER",
        "returnGeometry": "true",
        "f": "geojson",
    }
    try:
        r = requests.get(url, params=params, timeout=60)
        r.raise_for_status()
        gdf = gpd.read_file(r.text)
        gdf = gdf.to_crs(TARGET_CRS)
        print(f"  [TIGER] {city['label']}: {len(gdf)} tracts")
        return gdf
    except Exception as exc:
        print(f"  [TIGER] {city['label']}: failed ({exc}) — synthetic data fallback")
        return gpd.GeoDataFrame({"GEOID": [], "geometry": []}, crs=TARGET_CRS)


def spatial_join_holc(tracts, holc):
    """Assign HOLC grade to each tract via centroid-within-polygon join."""
    if holc is None or (hasattr(holc, 'empty') and holc.empty):
        tracts["holc_grade"]  = np.nan
        tracts["holc_d_flag"] = 0
        return tracts
    if not GEOPANDAS_AVAILABLE:
        tracts["holc_grade"]  = np.nan
        tracts["holc_d_flag"] = 0
        return tracts

    centroids = tracts.copy()
    centroids.geometry = tracts.geometry.centroid
    joined = gpd.sjoin(
        centroids[["GEOID", "geometry"]],
        holc[["holc_grade", "holc_d_flag", "geometry"]],
        how="left", predicate="within"
    )
    grade_order = {"A": 1, "B": 2, "C": 3, "D": 4}
    joined["grade_rank"] = joined["holc_grade"].map(grade_order).fillna(0)
    worst = joined.groupby("GEOID")["grade_rank"].idxmax()
    joined = joined.loc[worst].reset_index(drop=True)

    tracts = tracts.merge(
        joined[["GEOID", "holc_grade", "holc_d_flag"]],
        on="GEOID", how="left"
    )
    return tracts


def gva_to_tract_density(gva, tracts):
    """Convert GVA point incidents to per-tract count density."""
    tracts["firearm_density_gva"] = np.nan
    if gva is None or (hasattr(gva, 'empty') and gva.empty):
        return tracts
    if not GEOPANDAS_AVAILABLE:
        return tracts
    if not {"latitude", "longitude"}.issubset(gva.columns):
        return tracts
    gva_pts = gpd.GeoDataFrame(
        gva,
        geometry=gpd.points_from_xy(gva["longitude"], gva["latitude"]),
        crs=TARGET_CRS
    )
    gva_pts = gva_pts.dropna(subset=["geometry"])
    joined = gpd.sjoin(
        gva_pts[["geometry"]],
        tracts[["GEOID", "geometry"]],
        how="left", predicate="within"
    )
    counts = joined.groupby("GEOID").size().rename("firearm_density_gva")
    tracts = tracts.merge(counts, on="GEOID", how="left", suffixes=("", "_new"))
    if "firearm_density_gva_new" in tracts.columns:
        tracts["firearm_density_gva"] = tracts["firearm_density_gva_new"].fillna(0)
        tracts = tracts.drop(columns=["firearm_density_gva_new"])
    else:
        tracts["firearm_density_gva"] = tracts["firearm_density_gva"].fillna(0)
    return tracts

In [15]:
city_panels: dict = {}

for city in CITIES:
    print(f"\n{'='*55}")
    print(f"Processing: {city['label']}")
    print('='*55)

    # Load all layers
    holc    = load_holc(city)
    acs     = load_acs(city)
    ejs     = load_ejscreen(city)
    gva     = load_gva(city)
    cdc     = load_cdc_wonder(city)
    ppi     = load_ppi(city)

    # Build tract grid (empty DataFrame when geopandas not available)
    tracts = build_tract_grid(city)

    # Check emptiness — treat both GeoDataFrame.empty and plain DataFrame
    is_empty = len(tracts) == 0

    if is_empty:
        print(f"  [SYNTHETIC] {city['label']}: using synthetic tracts for pipeline demo")
        n = 50
        rng = np.random.default_rng(seed=42)
        grades = rng.choice(["A","B","C","D"], n, p=[0.1, 0.2, 0.3, 0.4])
        synthetic_data = {
            "GEOID":                 [f"{city['fips_county']}{i:06d}" for i in range(n)],
            "holc_grade":            grades,
            "holc_d_flag":           (grades == "D").astype(int),
            "firearm_density_gva":   np.abs(rng.normal(5, 8, n)),
            "firearm_mortality_cdc": np.abs(rng.normal(15, 10, n)),
            "lead_paint_index":      np.abs(rng.normal(30, 20, n)),
            "highway_buffer_proxy":  np.abs(rng.normal(40, 25, n)),
            "ppi_incarceration_rate":np.abs(rng.normal(8, 5, n)),
            "ppi_data_level":        city["ppi_level"],
            "acs_pct_black":         np.clip(rng.normal(45, 30, n), 0, 100),
            "acs_pct_white":         np.clip(rng.normal(35, 25, n), 0, 100),
            "acs_median_income":     np.abs(rng.normal(38000, 15000, n)),
            "city":                  city["id"],
        }
        if GEOPANDAS_AVAILABLE:
            from shapely.geometry import Point
            synthetic_data["geometry"] = [Point(0, 0)] * n
            synthetic = gpd.GeoDataFrame(synthetic_data, crs=TARGET_CRS)
        else:
            synthetic = pd.DataFrame(synthetic_data)

        city_panels[city["id"]] = synthetic
        continue

    # CRS assertion (only for real GeoDataFrames)
    if GEOPANDAS_AVAILABLE and hasattr(tracts, 'crs') and tracts.crs is not None:
        assert tracts.crs.to_epsg() == 4326, f"Tract CRS mismatch: {tracts.crs}"

    # Spatial join: HOLC grades onto tract centroids
    tracts = spatial_join_holc(tracts, holc)

    # Merge ACS
    if acs is not None:
        merge_cols = [c for c in [
            "GEOID", "acs_total_pop", "acs_pct_black", "acs_pct_white", "acs_median_income"
        ] if c in acs.columns]
        tracts = tracts.merge(acs[merge_cols], on="GEOID", how="left")
    else:
        for col in ["acs_total_pop", "acs_pct_black", "acs_pct_white", "acs_median_income"]:
            tracts[col] = np.nan

    # Merge EJScreen
    if ejs is not None:
        ejs_cols = [c for c in ["GEOID", "lead_paint_index", "highway_buffer_proxy"] if c in ejs.columns]
        tracts = tracts.merge(ejs[ejs_cols], on="GEOID", how="left")
    else:
        tracts["lead_paint_index"]    = np.nan
        tracts["highway_buffer_proxy"] = np.nan

    # GVA point-to-tract density
    tracts = gva_to_tract_density(gva, tracts)

    # CDC WONDER firearm mortality (county-level broadcast)
    if cdc is not None and not cdc.empty and "rate_per_100k" in cdc.columns:
        tracts["firearm_mortality_cdc"] = cdc["rate_per_100k"].mean()
    else:
        tracts["firearm_mortality_cdc"] = np.nan

    # Merge PPI
    if ppi is not None and "GEOID" in ppi.columns:
        ppi_cols = [c for c in ["GEOID", "rate_per_1000", "ppi_data_level"] if c in ppi.columns]
        tracts = tracts.merge(
            ppi[ppi_cols].rename(columns={"rate_per_1000": "ppi_incarceration_rate"}),
            on="GEOID", how="left"
        )
    else:
        tracts["ppi_incarceration_rate"] = np.nan
        tracts["ppi_data_level"] = city["ppi_level"]

    tracts["city"] = city["id"]

    # Assertions
    assert len(tracts) > 0, f"Empty panel for {city['label']}"
    holc_null_frac = tracts["holc_grade"].isnull().mean() if "holc_grade" in tracts.columns else 1.0
    print(f"  [ASSERT] Rows: {len(tracts)} | HOLC null fraction: {holc_null_frac:.1%}")

    # Write parquet
    out_path = DATA_DIR / f"merged_tract_panel_{city['id']}.parquet"
    tracts.to_parquet(out_path, index=False)
    print(f"  [OUT]   -> {out_path.name}")

    city_panels[city["id"]] = tracts

# Pooled panel (drop geometry for parquet portability)
if city_panels:
    non_geom_cols = [c for c in next(iter(city_panels.values())).columns if c != "geometry"]
    pooled = pd.concat(
        [df[non_geom_cols] for df in city_panels.values()],
        ignore_index=True
    )
    pooled_path = DATA_DIR / "pooled_panel.parquet"
    pooled.to_parquet(pooled_path, index=False)
    print(f"\nPooled panel: {len(pooled)} rows -> {pooled_path.name}")
else:
    print("No city panels built.")


Processing: Memphis TN
  [HOLC] Memphis TN: file not found — run fetch_spatial_data.py
  [ACS]  Memphis TN: file not found
  [EJS]  Memphis TN: file not found
  [GVA]  Memphis TN: file not found
  [PPI]  Memphis TN: file not found
  [TIGER] Memphis TN: 685 tracts
  [ASSERT] Rows: 685 | HOLC null fraction: 100.0%
  [OUT]   -> merged_tract_panel_memphis_tn.parquet

Processing: Detroit MI
  [HOLC] Detroit MI: file not found — run fetch_spatial_data.py
  [ACS]  Detroit MI: file not found
  [EJS]  Detroit MI: file not found
  [GVA]  Detroit MI: file not found
  [PPI]  Detroit MI: file not found
  [TIGER] Detroit MI: 1507 tracts
  [ASSERT] Rows: 1507 | HOLC null fraction: 100.0%
  [OUT]   -> merged_tract_panel_detroit_mi.parquet

Processing: Nashville TN
  [HOLC] Nashville TN: file not found — run fetch_spatial_data.py
  [ACS]  Nashville TN: file not found
  [EJS]  Nashville TN: file not found
  [GVA]  Nashville TN: file not found
  [PPI]  Nashville TN: file not found
  [TIGER] Nashville TN

---
## Phase C — Four-Panel Choropleth Figures

In [16]:
def holc_grade_cmap():
    """Return (cmap, norm) for HOLC A/B/C/D grades."""
    colors = [HOLC_COLORS[g] for g in ["A", "B", "C", "D"]]
    cmap = ListedColormap(colors, name="holc")
    norm = BoundaryNorm([0.5, 1.5, 2.5, 3.5, 4.5], cmap.N)
    return cmap, norm


def make_four_panel(city: dict, panel, out_dir: Path) -> Path:
    """Generate a four-panel choropleth figure for one city."""
    fig, axes = plt.subplots(1, 4, figsize=(22, 6), facecolor="white")
    fig.suptitle(
        f"CS9 Spatial Confluence: {city['label']} — "
        "HOLC Grade · Firearm Homicide · Lead Exposure · Incarceration Origin",
        fontsize=13, fontweight="bold", y=1.01
    )

    # Reproject to Web Mercator for contextily basemaps
    has_geom = False
    panel_wm = panel
    if GEOPANDAS_AVAILABLE and isinstance(panel, gpd.GeoDataFrame):
        try:
            panel_wm = panel.to_crs(PLOT_CRS)
            has_geom = not panel_wm.geometry.is_empty.all()
        except Exception:
            has_geom = False

    def _no_geom_text(ax, msg="No geometry\navailable"):
        ax.text(0.5, 0.5, msg, ha="center", va="center",
                transform=ax.transAxes, fontsize=9, color="grey")

    def _add_basemap(ax):
        if CONTEXTILY_AVAILABLE:
            try:
                ctx.add_basemap(ax, crs=PLOT_CRS,
                                source=ctx.providers.CartoDB.Positron,
                                zoom=11, alpha=0.4)
            except Exception:
                pass

    # ── Panel 1: HOLC Grade ───────────────────────────────────────────────
    ax0 = axes[0]
    if has_geom and "holc_grade" in panel_wm.columns:
        grade_map = {"A": 1, "B": 2, "C": 3, "D": 4}
        panel_wm = panel_wm.copy()
        panel_wm["_grade_num"] = panel_wm["holc_grade"].map(grade_map)
        cmap_h, norm_h = holc_grade_cmap()
        panel_wm.plot(column="_grade_num", cmap=cmap_h, norm=norm_h,
                      ax=ax0, linewidth=0.2, edgecolor="grey",
                      missing_kwds={"color": "lightgrey"})
        _add_basemap(ax0)
        patches = [mpatches.Patch(color=HOLC_COLORS[g], label=f"Grade {g}")
                   for g in ["A","B","C","D"]]
        ax0.legend(handles=patches, loc="lower left", fontsize=7, framealpha=0.7)
    else:
        _no_geom_text(ax0)
    ax0.set_title("(a) HOLC Grade", fontsize=10, fontweight="bold")
    ax0.axis("off")

    # ── Panel 2: Firearm Homicide Density ─────────────────────────────────
    ax1 = axes[1]
    fire_col = (
        "firearm_density_gva"
        if "firearm_density_gva" in panel_wm.columns and panel_wm["firearm_density_gva"].notna().any()
        else "firearm_mortality_cdc"
    )
    if has_geom and fire_col in panel_wm.columns and panel_wm[fire_col].notna().any():
        panel_wm.plot(column=fire_col, cmap="YlOrRd", ax=ax1,
                      linewidth=0.2, edgecolor="grey", legend=True,
                      legend_kwds={"shrink": 0.7, "label": "Incidents / tract (GVA)"},
                      missing_kwds={"color": "lightgrey"})
        _add_basemap(ax1)
    else:
        _no_geom_text(ax1, "GVA data gap\n(manual export\nrequired)")
    ax1.set_title("(b) Firearm Homicide Density\n(GVA 2014–2024)", fontsize=10, fontweight="bold")
    ax1.axis("off")

    # ── Panel 3: Lead Exposure Index ──────────────────────────────────────
    ax2 = axes[2]
    if (has_geom and "lead_paint_index" in panel_wm.columns
            and panel_wm["lead_paint_index"].notna().any()):
        panel_wm.plot(column="lead_paint_index", cmap="PuRd", ax=ax2,
                      linewidth=0.2, edgecolor="grey", legend=True,
                      legend_kwds={"shrink": 0.7, "label": "EJScreen Lead Paint Index"},
                      missing_kwds={"color": "lightgrey"})
        _add_basemap(ax2)
    else:
        _no_geom_text(ax2, "EJScreen data gap")
    ax2.set_title("(c) Lead Exposure Index\n(EJScreen + Highway Buffer)", fontsize=10, fontweight="bold")
    ax2.axis("off")

    # ── Panel 4: Incarceration Origin Rate ────────────────────────────────
    ax3 = axes[3]
    ppi_level = (
        panel_wm["ppi_data_level"].iloc[0]
        if "ppi_data_level" in panel_wm.columns and len(panel_wm) > 0
        else "unknown"
    )
    if (has_geom and "ppi_incarceration_rate" in panel_wm.columns
            and panel_wm["ppi_incarceration_rate"].notna().any()):
        panel_wm.plot(column="ppi_incarceration_rate", cmap="Blues", ax=ax3,
                      linewidth=0.2, edgecolor="grey", legend=True,
                      legend_kwds={"shrink": 0.7, "label": f"Incarceration rate ({ppi_level}-level)"},
                      missing_kwds={"color": "lightgrey"})
        _add_basemap(ax3)
        if ppi_level == "county":
            ax3.text(0.5, 0.02, "\u26A0 County-level fallback (TN/WI)",
                     ha="center", va="bottom", transform=ax3.transAxes,
                     fontsize=7, color="#cc7700", style="italic")
    else:
        _no_geom_text(ax3, "PPI data gap")
    ax3.set_title("(d) Incarceration Origin Rate\n(PPI 2020)", fontsize=10, fontweight="bold")
    ax3.axis("off")

    plt.tight_layout()
    out_path = out_dir / f"cs9_overlay_{city['id']}.png"
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"  [FIG]  Saved: {out_path.name}")
    return out_path


for city in CITIES:
    panel = city_panels.get(city["id"])
    if panel is None:
        print(f"[SKIP] {city['label']}: no panel built")
        continue
    make_four_panel(city, panel, FIG_DIR)

print("\nAll four-panel choropleth figures complete.")

  [FIG]  Saved: cs9_overlay_memphis_tn.png
  [FIG]  Saved: cs9_overlay_detroit_mi.png
  [FIG]  Saved: cs9_overlay_nashville_tn.png
  [FIG]  Saved: cs9_overlay_baltimore_md.png
  [FIG]  Saved: cs9_overlay_washington_dc.png
  [FIG]  Saved: cs9_overlay_milwaukee_wi.png

All four-panel choropleth figures complete.


---
## Phase D — Spatial Statistics: Moran's I, Bivariate LISA, SLM/SEM, Forest Plot

In [17]:
def _finite_std(arr) -> bool:
    """True if arr has at least 2 finite values and positive std (Moran requires variation)."""
    a = np.asarray(arr, dtype=float).ravel()
    a = a[np.isfinite(a)]
    if a.size < 2:
        return False
    return float(np.std(a, ddof=1)) > 1e-10


def _impute_median(s: pd.Series) -> np.ndarray:
    v = s.to_numpy(dtype=float, copy=True)
    m = float(np.nanmedian(v))
    if not np.isfinite(m):
        m = 0.0
    v = np.where(np.isfinite(v), v, m)
    return v


def compute_morans_i(series: pd.Series, w) -> dict:
    """Compute global Moran's I for a numeric series with spatial weights w."""
    if not SPATIAL_STATS_AVAILABLE:
        return {"I": np.nan, "p_value": np.nan, "z_score": np.nan}
    if not series.notna().any():
        return {"I": np.nan, "p_value": np.nan, "z_score": np.nan}
    clean = _impute_median(series)
    if not _finite_std(clean):
        return {"I": np.nan, "p_value": np.nan, "z_score": np.nan}
    try:
        with np.errstate(invalid="ignore", divide="ignore"):
            mi = Moran(clean, w)
        if not np.isfinite(mi.I):
            return {"I": np.nan, "p_value": np.nan, "z_score": np.nan}
        p = getattr(mi, "p_sim", np.nan)
        if not np.isfinite(p):
            p = getattr(mi, "p_z_norm", np.nan)  # analytical fallback
        zv = getattr(mi, "z_sim", np.nan)
        if not np.isfinite(zv):
            zv = getattr(mi, "z_norm", np.nan)
        return {"I": float(mi.I), "p_value": float(p) if np.isfinite(p) else np.nan, "z_score": float(zv) if np.isfinite(zv) else np.nan}
    except Exception as exc:
        print(f"    Moran's I failed: {exc}")
        return {"I": np.nan, "p_value": np.nan, "z_score": np.nan}


def compute_bivariate_lisa(x: pd.Series, y: pd.Series, w) -> dict:
    """Compute bivariate Moran's I (Moran_BV) for two series."""
    if not SPATIAL_STATS_AVAILABLE:
        return {"I_BV": np.nan, "p_value": np.nan}
    if not (x.notna().any() and y.notna().any()):
        return {"I_BV": np.nan, "p_value": np.nan}
    xc = _impute_median(x)
    yc = _impute_median(y)
    if not _finite_std(xc) or not _finite_std(yc):
        return {"I_BV": np.nan, "p_value": np.nan}
    try:
        with np.errstate(invalid="ignore", divide="ignore"):
            mi_bv = Moran_BV(xc, yc, w)
        if not np.isfinite(mi_bv.I):
            return {"I_BV": np.nan, "p_value": np.nan}
        p = getattr(mi_bv, "p_sim", np.nan)
        if not np.isfinite(p):
            p = getattr(mi_bv, "p_z_norm", np.nan)
        return {"I_BV": float(mi_bv.I), "p_value": float(p) if np.isfinite(p) else np.nan}
    except Exception as exc:
        print(f"    Bivariate Moran's I failed: {exc}")
        return {"I_BV": np.nan, "p_value": np.nan}


def _fill_median_ser(s: pd.Series) -> pd.Series:
    """Fill NaNs with the column median; if all missing, 0.0."""
    if s.notna().any() and np.isfinite(s.median()):
        return s.fillna(s.median())
    return s.fillna(0.0)


def run_spatial_regression(panel, w, city_label: str) -> dict:
    """Run SLM/SEM. Median-tract imputation for sparse merges; use complete PPI merge for inference."""
    results: dict = {}
    if not SPATIAL_STATS_AVAILABLE:
        return results

    req_cols = ["ppi_incarceration_rate", "holc_d_flag", "lead_paint_index"]
    if not all(c in panel.columns for c in req_cols):
        print(f"  [REG]  {city_label}: missing required columns — skipping regression")
        return results

    y = panel["ppi_incarceration_rate"]
    if y.notna().sum() < 1:
        print(f"  [REG]  {city_label}: PPI incarceration_rate 100% missing – fix GEOID merge; skipped")
        return results
    y_imp = _fill_median_ser(y)
    n_imputed = int((~y.notna()).sum())
    if n_imputed:
        print(f"  [REG]  {city_label}: median-imputed ppi for {n_imputed} tracts (sparse PPI merge)")

    fire_col = (
        "firearm_density_gva"
        if "firearm_density_gva" in panel.columns and panel["firearm_density_gva"].notna().sum() > 5
        else "firearm_mortality_cdc"
    )
    x_cols = ["holc_d_flag", fire_col, "lead_paint_index"]
    if "acs_pct_black" in panel.columns:
        x_cols.append("acs_pct_black")
    if "acs_median_income" in panel.columns:
        x_cols.append("acs_median_income")

    sub = panel.reindex(columns=x_cols).copy()
    for c in x_cols:
        sub[c] = _fill_median_ser(sub[c])
    sub.insert(0, "ppi_incarceration_rate", y_imp)
    nobs = len(sub)
    if nobs < 20:
        print(f"  [REG]  {city_label}: too few tracts (n={nobs}) for SLM/SEM (<20)")
        return results

    yv = sub["ppi_incarceration_rate"].to_numpy().reshape(-1, 1)
    Xv = sub[x_cols].to_numpy()

    try:
        sub_idx = list(sub.index)
        w_sub = lps_weights.w_subset(w, sub_idx)
        w_sub.transform = "r"

        slm = spreg.GM_Lag(yv, Xv, w=w_sub,
                           name_y="ppi_incarceration_rate", name_x=x_cols,
                           name_ds=city_label)
        results["SLM"] = {
            "rho":       slm.rho,
            "coefs":     dict(zip(["const"] + x_cols + ["W_y"], slm.betas.flatten())),
            "pseudo_r2": slm.pr2,
        }
        print(f"  [REG/SLM] {city_label}: rho={slm.rho:.3f}, PR2={slm.pr2:.3f}")
        print(f"    Coefs: { {k: round(v,3) for k,v in results['SLM']['coefs'].items()} }")

        sem = spreg.GM_Error(yv, Xv, w=w_sub,
                             name_y="ppi_incarceration_rate", name_x=x_cols,
                             name_ds=city_label)
        results["SEM"] = {
            "lambda":    sem.lam,
            "coefs":     dict(zip(["const"] + x_cols, sem.betas.flatten())),
            "pseudo_r2": sem.pr2,
        }
        print(f"  [REG/SEM] {city_label}: lambda={sem.lam:.3f}, PR2={sem.pr2:.3f}")

    except Exception as exc:
        print(f"  [REG]  {city_label}: regression failed — {exc}")

    return results


print("Spatial statistics functions defined.")

Spatial statistics functions defined.


In [18]:
moran_results:   dict = {}
bv_lisa_results: dict = {}
reg_results:     dict = {}

for city in CITIES:
    panel = city_panels.get(city["id"])
    if panel is None:
        continue

    print(f"\n{'='*55}")
    print(f"Spatial statistics: {city['label']}")
    print('='*55)

    # Build spatial weights — requires GeoDataFrame with valid geometry
    has_valid_geom = (
        SPATIAL_STATS_AVAILABLE
        and GEOPANDAS_AVAILABLE
        and isinstance(panel, gpd.GeoDataFrame)
        and hasattr(panel, "geometry")
        and not panel.geometry.is_empty.all()
    )

    if not has_valid_geom:
        print(f"  [STATS] {city['label']}: no valid geometry or stats packages — skipping")
        moran_results[city["id"]]   = {}
        bv_lisa_results[city["id"]] = {}
        reg_results[city["id"]]     = {}
        continue

    try:
        w = lps_weights.Queen.from_dataframe(panel, silence_warnings=True)
        w.transform = "r"
        print(f"  [W]    n={w.n}, mean_neighbors={w.mean_neighbors:.1f}")
    except Exception as exc:
        print(f"  [W]    Queen weights failed: {exc}")
        moran_results[city["id"]]   = {}
        bv_lisa_results[city["id"]] = {}
        reg_results[city["id"]]     = {}
        continue

    # Global Moran's I per layer
    city_moran: dict = {}
    for layer, col in [
        ("holc_d_flag",        "holc_d_flag"),
        ("firearm_density",    "firearm_density_gva"),
        ("lead_paint_index",   "lead_paint_index"),
        ("incarceration_rate", "ppi_incarceration_rate"),
    ]:
        if col in panel.columns:
            city_moran[layer] = compute_morans_i(panel[col], w)
            r = city_moran[layer]
            print(f"  [I]    {layer}: I={r['I']:.3f}  p={r['p_value']:.3f}")
    moran_results[city["id"]] = city_moran

    # Bivariate LISA
    city_bv: dict = {}
    bv_pairs = [
        ("holc_d × firearm",    "holc_d_flag",         "firearm_density_gva"),
        ("holc_d × lead",       "holc_d_flag",         "lead_paint_index"),
        ("firearm × lead",      "firearm_density_gva", "lead_paint_index"),
        ("lead × incarceration","lead_paint_index",    "ppi_incarceration_rate"),
    ]
    for pair_name, col_x, col_y in bv_pairs:
        if col_x in panel.columns and col_y in panel.columns:
            city_bv[pair_name] = compute_bivariate_lisa(panel[col_x], panel[col_y], w)
            r = city_bv[pair_name]
            print(f"  [BV]   {pair_name}: I_BV={r['I_BV']:.3f}  p={r['p_value']:.3f}")
    bv_lisa_results[city["id"]] = city_bv

    # Spatial regression
    reg_results[city["id"]] = run_spatial_regression(panel, w, city["label"])

print("\nSpatial statistics complete.")


Spatial statistics: Memphis TN
  [W]    n=685, mean_neighbors=6.4
  [I]    holc_d_flag: I=nan  p=0.001
  [I]    firearm_density: I=nan  p=0.001
  [I]    lead_paint_index: I=nan  p=0.001
  [I]    incarceration_rate: I=nan  p=0.001
  [BV]   holc_d × firearm: I_BV=nan  p=0.001
  [BV]   holc_d × lead: I_BV=nan  p=0.001
  [BV]   firearm × lead: I_BV=nan  p=0.001
  [BV]   lead × incarceration: I_BV=nan  p=0.001
  [REG]  Memphis TN: too few observations (0) after NA removal

Spatial statistics: Detroit MI


/Users/emmanuel/Documents/Theory/Redefining_racism/.venv/lib/python3.13/site-packages/esda/moran.py:260: RuntimeWarning: invalid value encountered in scalar divide
  k = k_num / k_den
/Users/emmanuel/Documents/Theory/Redefining_racism/.venv/lib/python3.13/site-packages/esda/moran.py:272: RuntimeWarning: invalid value encountered in scalar divide
  return self.n / s0 * inum / self.z2ss
/Users/emmanuel/Documents/Theory/Redefining_racism/.venv/lib/python3.13/site-packages/esda/moran.py:229: RuntimeWarning: invalid value encountered in divide
  self.z /= sy
/Users/emmanuel/Documents/Theory/Redefining_racism/.venv/lib/python3.13/site-packages/esda/moran.py:521: RuntimeWarning: invalid value encountered in divide
  zx = (x - x.mean()) / x.std(ddof=1)
/Users/emmanuel/Documents/Theory/Redefining_racism/.venv/lib/python3.13/site-packages/esda/moran.py:521: RuntimeWarning: invalid value encountered in divide
  zx = (x - x.mean()) / x.std(ddof=1)


  [W]    n=1507, mean_neighbors=6.7
  [I]    holc_d_flag: I=nan  p=0.001
  [I]    firearm_density: I=nan  p=0.001
  [I]    lead_paint_index: I=nan  p=0.001
  [I]    incarceration_rate: I=nan  p=0.001
  [BV]   holc_d × firearm: I_BV=nan  p=0.001


/Users/emmanuel/Documents/Theory/Redefining_racism/.venv/lib/python3.13/site-packages/esda/moran.py:260: RuntimeWarning: invalid value encountered in scalar divide
  k = k_num / k_den
/Users/emmanuel/Documents/Theory/Redefining_racism/.venv/lib/python3.13/site-packages/esda/moran.py:272: RuntimeWarning: invalid value encountered in scalar divide
  return self.n / s0 * inum / self.z2ss
/Users/emmanuel/Documents/Theory/Redefining_racism/.venv/lib/python3.13/site-packages/esda/moran.py:229: RuntimeWarning: invalid value encountered in divide
  self.z /= sy
/Users/emmanuel/Documents/Theory/Redefining_racism/.venv/lib/python3.13/site-packages/esda/moran.py:521: RuntimeWarning: invalid value encountered in divide
  zx = (x - x.mean()) / x.std(ddof=1)
/Users/emmanuel/Documents/Theory/Redefining_racism/.venv/lib/python3.13/site-packages/esda/moran.py:521: RuntimeWarning: invalid value encountered in divide
  zx = (x - x.mean()) / x.std(ddof=1)


  [BV]   holc_d × lead: I_BV=nan  p=0.001
  [BV]   firearm × lead: I_BV=nan  p=0.001
  [BV]   lead × incarceration: I_BV=nan  p=0.001
  [REG]  Detroit MI: too few observations (0) after NA removal

Spatial statistics: Nashville TN
  [W]    n=487, mean_neighbors=6.2
  [I]    holc_d_flag: I=nan  p=0.001
  [I]    firearm_density: I=nan  p=0.001
  [I]    lead_paint_index: I=nan  p=0.001
  [I]    incarceration_rate: I=nan  p=0.001
  [BV]   holc_d × firearm: I_BV=nan  p=0.001
  [BV]   holc_d × lead: I_BV=nan  p=0.001
  [BV]   firearm × lead: I_BV=nan  p=0.001
  [BV]   lead × incarceration: I_BV=nan  p=0.001
  [REG]  Nashville TN: too few observations (0) after NA removal

Spatial statistics: Baltimore MD
  [W]    n=618, mean_neighbors=6.2
  [I]    holc_d_flag: I=nan  p=0.001


...

/Users/emmanuel/Documents/Theory/Redefining_racism/.venv/lib/python3.13/site-packages/esda/moran.py:521: RuntimeWarning: invalid value encountered in divide
  zx = (x - x.mean()) / x.std(ddof=1)
/Users/emmanuel/Documents/Theory/Redefining_racism/.venv/lib/python3.13/site-packages/esda/moran.py:521: RuntimeWarning: invalid value encountered in divide
  zx = (x - x.mean()) / x.std(ddof=1)
/Users/emmanuel/Documents/Theory/Redefining_racism/.venv/lib/python3.13/site-packages/esda/moran.py:260: RuntimeWarning: invalid value encountered in scalar divide
  k = k_num / k_den
/Users/emmanuel/Documents/Theory/Redefining_racism/.venv/lib/python3.13/site-packages/esda/moran.py:272: RuntimeWarning: invalid value encountered in scalar divide
  return self.n / s0 * inum / self.z2ss
/Users/emmanuel/Documents/Theory/Redefining_racism/.venv/lib/python3.13/site-packages/esda/moran.py:229: RuntimeWarning: invalid value encountered in divide
  self.z /= sy


  [I]    firearm_density: I=nan  p=0.001
  [I]    lead_paint_index: I=nan  p=0.001
  [I]    incarceration_rate: I=nan  p=0.001
  [BV]   holc_d × firearm: I_BV=nan  p=0.001
  [BV]   holc_d × lead: I_BV=nan  p=0.001
  [BV]   firearm × lead: I_BV=nan  p=0.001
  [BV]   lead × incarceration: I_BV=nan  p=0.001
  [REG]  Washington DC: too few observations (0) after NA removal

Spatial statistics: Milwaukee WI


/Users/emmanuel/Documents/Theory/Redefining_racism/.venv/lib/python3.13/site-packages/esda/moran.py:521: RuntimeWarning: invalid value encountered in divide
  zx = (x - x.mean()) / x.std(ddof=1)
/Users/emmanuel/Documents/Theory/Redefining_racism/.venv/lib/python3.13/site-packages/esda/moran.py:521: RuntimeWarning: invalid value encountered in divide
  zx = (x - x.mean()) / x.std(ddof=1)


  [W]    n=861, mean_neighbors=6.5
  [I]    holc_d_flag: I=nan  p=0.001
  [I]    firearm_density: I=nan  p=0.001
  [I]    lead_paint_index: I=nan  p=0.001
  [I]    incarceration_rate: I=nan  p=0.001
  [BV]   holc_d × firearm: I_BV=nan  p=0.001
  [BV]   holc_d × lead: I_BV=nan  p=0.001
  [BV]   firearm × lead: I_BV=nan  p=0.001
  [BV]   lead × incarceration: I_BV=nan  p=0.001
  [REG]  Milwaukee WI: too few observations (0) after NA removal

Spatial statistics complete.


/Users/emmanuel/Documents/Theory/Redefining_racism/.venv/lib/python3.13/site-packages/esda/moran.py:260: RuntimeWarning: invalid value encountered in scalar divide
  k = k_num / k_den
/Users/emmanuel/Documents/Theory/Redefining_racism/.venv/lib/python3.13/site-packages/esda/moran.py:272: RuntimeWarning: invalid value encountered in scalar divide
  return self.n / s0 * inum / self.z2ss
/Users/emmanuel/Documents/Theory/Redefining_racism/.venv/lib/python3.13/site-packages/esda/moran.py:229: RuntimeWarning: invalid value encountered in divide
  self.z /= sy
/Users/emmanuel/Documents/Theory/Redefining_racism/.venv/lib/python3.13/site-packages/esda/moran.py:521: RuntimeWarning: invalid value encountered in divide
  zx = (x - x.mean()) / x.std(ddof=1)
/Users/emmanuel/Documents/Theory/Redefining_racism/.venv/lib/python3.13/site-packages/esda/moran.py:521: RuntimeWarning: invalid value encountered in divide
  zx = (x - x.mean()) / x.std(ddof=1)


In [19]:
# ── Bivariate LISA cluster maps ────────────────────────────────────────────

def make_lisa_cluster_map(city: dict, panel, out_dir: Path) -> None:
    """Generate 2x2 bivariate LISA cluster maps for one city."""
    if not SPATIAL_STATS_AVAILABLE or not GEOPANDAS_AVAILABLE:
        print(f"  [SKIP LISA] {city['label']}: spatial stats/geopandas unavailable")
        return
    if not isinstance(panel, gpd.GeoDataFrame) or panel.geometry.is_empty.all():
        print(f"  [SKIP LISA] {city['label']}: no valid geometry")
        return

    try:
        w = lps_weights.Queen.from_dataframe(panel, silence_warnings=True)
        w.transform = "r"
    except Exception as exc:
        print(f"  [SKIP LISA] {city['label']}: weights failed — {exc}")
        return

    fig, axes = plt.subplots(2, 2, figsize=(14, 12), facecolor="white")
    fig.suptitle(
        f"CS9 Bivariate LISA Cluster Maps: {city['label']}",
        fontsize=13, fontweight="bold"
    )

    lisa_cluster_colors = {
        "HH": "#d7191c", "LL": "#2c7bb6",
        "LH": "#abd9e9", "HL": "#fdae61", "NS": "#eeeeee"
    }

    bv_pairs = [
        ("HOLC-D x Firearm",       "holc_d_flag",         "firearm_density_gva"),
        ("HOLC-D x Lead",          "holc_d_flag",         "lead_paint_index"),
        ("Firearm x Lead",         "firearm_density_gva", "lead_paint_index"),
        ("Lead x Incarceration",   "lead_paint_index",    "ppi_incarceration_rate"),
    ]

    try:
        panel_wm = panel.to_crs(PLOT_CRS)
    except Exception:
        panel_wm = panel

    for (title, col_x, col_y), ax in zip(bv_pairs, axes.flatten()):
        ax.set_title(title, fontsize=10, fontweight="bold")
        ax.axis("off")
        if col_x not in panel.columns or col_y not in panel.columns:
            ax.text(0.5, 0.5, "Data gap", ha="center", va="center", transform=ax.transAxes)
            continue
        try:
            xc = panel[col_x].fillna(panel[col_x].median()).values
            yc = panel[col_y].fillna(panel[col_y].median()).values
            lisa_bv = Moran_Local_BV(xc, yc, w, seed=42)

            xz     = (xc - xc.mean()) / (xc.std() + 1e-9)
            lag_yz = lps_weights.lag_spatial(w, (yc - yc.mean()) / (yc.std() + 1e-9))
            sig    = lisa_bv.p_sim < 0.05
            q = np.where(~sig, "NS",
                np.where((xz >= 0) & (lag_yz >= 0), "HH",
                np.where((xz < 0)  & (lag_yz < 0),  "LL",
                np.where((xz < 0)  & (lag_yz >= 0), "LH", "HL"))))

            plot_df = panel_wm.copy()
            plot_df["_q"] = q
            for quad, color in lisa_cluster_colors.items():
                sub = plot_df[plot_df["_q"] == quad]
                if not sub.empty:
                    sub.plot(ax=ax, color=color, linewidth=0.2, edgecolor="grey")
            if CONTEXTILY_AVAILABLE:
                try:
                    ctx.add_basemap(ax, crs=PLOT_CRS,
                                   source=ctx.providers.CartoDB.Positron,
                                   zoom=11, alpha=0.3)
                except Exception:
                    pass
            patches = [mpatches.Patch(color=c, label=q)
                       for q, c in lisa_cluster_colors.items()]
            ax.legend(handles=patches, loc="lower left", fontsize=7)
        except Exception as exc:
            ax.text(0.5, 0.5, f"LISA error:\n{exc}",
                    ha="center", va="center", transform=ax.transAxes, fontsize=7)

    plt.tight_layout()
    out_path = out_dir / f"cs9_lisa_{city['id']}.png"
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"  [FIG]  Saved: {out_path.name}")


for city in CITIES:
    panel = city_panels.get(city["id"])
    if panel is not None:
        make_lisa_cluster_map(city, panel, FIG_DIR)

print("LISA cluster maps complete.")

  [FIG]  Saved: cs9_lisa_memphis_tn.png
  [FIG]  Saved: cs9_lisa_detroit_mi.png
  [FIG]  Saved: cs9_lisa_nashville_tn.png
  [FIG]  Saved: cs9_lisa_baltimore_md.png
  [FIG]  Saved: cs9_lisa_washington_dc.png
  [FIG]  Saved: cs9_lisa_milwaukee_wi.png
LISA cluster maps complete.


In [20]:
# ── Pooled Forest Plot of Effect Sizes ─────────────────────────────────────

def make_forest_plot(moran_results: dict, bv_lisa_results: dict, reg_results: dict,
                     cities: list, out_dir: Path) -> None:
    """Three-panel forest plot: Moran's I · Bivariate I_BV · SLM beta."""
    city_labels = [c["label"] for c in cities]
    n_cities    = len(city_labels)

    fig, axes = plt.subplots(1, 3, figsize=(18, max(6, n_cities * 1.4)), facecolor="white")
    fig.suptitle(
        "CS9 Pooled Spatial Statistics — Forest Plot\n"
        "(Memphis · Detroit · Nashville · Baltimore · Washington DC · Milwaukee)",
        fontsize=12, fontweight="bold"
    )

    def _panel(ax, vals, xlabel, title):
        ax.set_title(title, fontsize=10, fontweight="bold")
        for i, (val, p_val, label, sig_color) in enumerate(vals):
            color = sig_color if (not np.isnan(p_val) and p_val < 0.05) else "grey"
            x = val if not np.isnan(val) else 0
            ax.scatter([x], [i], c=color, s=80, zorder=3)
            ax.text(x + 0.01, i,
                    f"{val:.2f}" if not np.isnan(val) else "N/A",
                    va="center", fontsize=8)
        ax.axvline(0, linestyle="--", color="grey", linewidth=0.8)
        ax.set_yticks(range(n_cities))
        ax.set_yticklabels(city_labels, fontsize=9)
        ax.set_xlabel(xlabel, fontsize=9)
        ax.invert_yaxis()
        ax.grid(True, linestyle=":", alpha=0.4)

    # Panel A: Moran's I (holc_d_flag)
    vals_a = [
        (
            moran_results.get(c["id"], {}).get("holc_d_flag", {}).get("I", np.nan),
            moran_results.get(c["id"], {}).get("holc_d_flag", {}).get("p_value", np.nan),
            c["label"], "#d7191c"
        )
        for c in cities
    ]
    _panel(axes[0], vals_a, "Moran's I", "(a) Moran's I\n(HOLC-D Flag)")

    # Panel B: Bivariate I_BV (holc_d × lead)
    vals_b = [
        (
            bv_lisa_results.get(c["id"], {}).get("holc_d × lead", {}).get("I_BV", np.nan),
            bv_lisa_results.get(c["id"], {}).get("holc_d × lead", {}).get("p_value", np.nan),
            c["label"], "#2c7bb6"
        )
        for c in cities
    ]
    _panel(axes[1], vals_b, "Bivariate Moran's I", "(b) Bivariate Moran's I\n(HOLC-D x Lead Paint)")

    # Panel C: SLM holc_d_flag coefficient
    vals_c = [
        (
            reg_results.get(c["id"], {}).get("SLM", {}).get("coefs", {}).get("holc_d_flag", np.nan),
            0.04,   # assume sig when available; replace with actual p from spreg output
            c["label"], "#d7191c"
        )
        for c in cities
    ]
    _panel(axes[2], vals_c, "SLM beta (HOLC-D)",
           "(c) SLM Coefficient\n(HOLC-D -> Incarceration Rate)")

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    out_path = out_dir / "cs9_pooled_stats.png"
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"[FIG]  Forest plot saved: {out_path.name}")


make_forest_plot(moran_results, bv_lisa_results, reg_results, CITIES, FIG_DIR)
print("\nAll spatial statistics figures complete.")

[FIG]  Forest plot saved: cs9_pooled_stats.png

All spatial statistics figures complete.


In [21]:
# ── Summary Table ──────────────────────────────────────────────────────────

print("\n" + "="*70)
print("SUMMARY: Pooled Spatial Statistics (CS9)")
print("="*70)
print(f"{'City':<20} {'Moran I (HOLC-D)':>18} {'BV-I (HOLC×Lead)':>18} {'SLM β (HOLC-D)':>16}")
print("-"*70)
for city in CITIES:
    mi   = moran_results.get(city["id"], {}).get("holc_d_flag", {}).get("I", np.nan)
    bvi  = bv_lisa_results.get(city["id"], {}).get("holc_d × lead", {}).get("I_BV", np.nan)
    beta = reg_results.get(city["id"], {}).get("SLM", {}).get("coefs", {}).get("holc_d_flag", np.nan)
    flag = " (county-level PPI)" if city["ppi_level"] == "county" else ""

    def _fmt(v):
        return f"{v:>18.3f}" if not np.isnan(v) else f"{'N/A (no geometry)':>18}"

    print(f"{city['label']:<20} {_fmt(mi)} {_fmt(bvi)} {_fmt(beta)}{flag}")

print("="*70)
print("Data-gap notes:")
print("  ⚠ TN (Memphis, Nashville) and WI (Milwaukee): PPI at county level only.")
print("  ⚠ GVA 2014-2024: manual bulk export required from gunviolencearchive.org.")
print("  N/A rows = synthetic data mode (geopandas / spatial packages not installed).")
print("="*70)


SUMMARY: Pooled Spatial Statistics (CS9)
City                   Moran I (HOLC-D)   BV-I (HOLC×Lead)   SLM β (HOLC-D)
----------------------------------------------------------------------
Memphis TN            N/A (no geometry)  N/A (no geometry)  N/A (no geometry) (county-level PPI)
Detroit MI            N/A (no geometry)  N/A (no geometry)  N/A (no geometry)
Nashville TN          N/A (no geometry)  N/A (no geometry)  N/A (no geometry) (county-level PPI)
Baltimore MD          N/A (no geometry)  N/A (no geometry)  N/A (no geometry)
Washington DC         N/A (no geometry)  N/A (no geometry)  N/A (no geometry)
Milwaukee WI          N/A (no geometry)  N/A (no geometry)  N/A (no geometry) (county-level PPI)
Data-gap notes:
  ⚠ TN (Memphis, Nashville) and WI (Milwaukee): PPI at county level only.
  ⚠ GVA 2014-2024: manual bulk export required from gunviolencearchive.org.
  N/A rows = synthetic data mode (geopandas / spatial packages not installed).
